In [1]:
import pandas as pd
from hdx_client import HdxClient

from processing.food_prices_processing import *
from processing.rainfall_processing import *

## Test food prices

In [3]:
read_food = read_food_prices(False)
clean_food = get_clean_data(False)

INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: C:\Users\evely\.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:WFP Ingest:download=False: Reading local file ../data/hdx\Sudan - Food Prices.csv
INFO:WFP Ingest:download=False: Reading local file ../data/hdx\South Sudan - Food Prices.csv


## Test rainfall

In [2]:
df = read_rainfall(download=False, remove_abyei=False)

INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: C:\Users\evely\.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\ssd-rainfall-subnat-full.csv


South Sudan PCODE sample: ['SS01', 'SS01', 'SS01', 'SS01', 'SS01']
South Sudan columns: ['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg', 'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version', 'region']


In [4]:
print("Is Abyei in region?:", "Abyei" in df["region"].values)

Is Abyei in region?: False


In [8]:
# Initialize HDX Client
hdx = HdxClient()

# 1. Read Sudan rainfall dataset
rainfall_sudan = hdx.get_data(
    dataset_name="sdn-rainfall-subnational",
    file_name="sdn-rainfall-subnat-full",
    file_type="csv",
    download=False,
)

# 2. Search entire Sudan dataframe for "abyei" across all columns
abyei_sudan = rainfall_sudan[
    rainfall_sudan.astype(str)
    .apply(lambda row: row.str.contains("abyei", case=False))
    .any(axis=1)
]

# 3. Print results
print("Found Abyei rows in Sudan:", len(abyei_sudan))

INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv


Found Abyei rows in Sudan: 0


In [9]:
from ingest.hdx_client import HdxClient

hdx = HdxClient()

# Load Sudan
sudan = hdx.get_data(
    "sdn-rainfall-subnational", "sdn-rainfall-subnat-full", "csv", download=False
)
print("Sudan adm_levels present:", sudan["adm_level"].unique())
print(
    "Sudan PCODEs ending in 19 or containing Abyei PCODE patterns:",
    [p for p in sudan["PCODE"].unique() if "19" in str(p)],
)

# Load South Sudan
ss = hdx.get_data(
    "ssd-rainfall-subnational", "ssd-rainfall-subnat-full", "csv", download=False
)
print("\nSouth Sudan adm_levels present:", ss["adm_level"].unique())
print(
    "South Sudan PCODEs starting with SS03 or SS04 (Warrap/Unity sub-districts):",
    [
        p
        for p in ss["PCODE"].unique()
        if str(p).startswith(("SS03", "SS04", "SS80", "SS19"))
    ],
)

INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\ssd-rainfall-subnat-full.csv


Sudan adm_levels present: [1 2]
Sudan PCODEs ending in 19 or containing Abyei PCODE patterns: ['SD17019', 'SD02119']

South Sudan adm_levels present: [1 2]
South Sudan PCODEs starting with SS03 or SS04 (Warrap/Unity sub-districts): ['SS03', 'SS04', 'SS0301', 'SS0401', 'SS0302', 'SS0303', 'SS0304', 'SS0402', 'SS0305', 'SS0306', 'SS0307', 'SS0308', 'SS0309', 'SS0403', 'SS0404', 'SS0405', 'SS0310', 'SS0311', 'SS0406', 'SS0407', 'SS0408']


In [6]:
# Load raw South Sudan dataset
hdx = HdxClient()
rainfall_ss = hdx.get_data(
    dataset_name="ssd-rainfall-subnational",
    file_name="ssd-rainfall-subnat-full",
    file_type="csv",
    download=False,
)

# Filter for admin level 1
rainfall_ss_admin1 = rainfall_ss[rainfall_ss["adm_level"] == 1].copy()

# Identify the name column (usually adm1_name, admin1, or name)
name_col = next(
    (
        c
        for c in ["adm1_name", "admin1", "name", "region"]
        if c in rainfall_ss_admin1.columns
    ),
    None,
)

# Display unique PCODEs along with their area names
if name_col:
    pcode_df = (
        rainfall_ss_admin1[["PCODE", name_col]].drop_duplicates().reset_index(drop=True)
    )
    print(pcode_df)
else:
    print("Unique PCODEs:", rainfall_ss_admin1["PCODE"].unique())

INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\ssd-rainfall-subnat-full.csv


Unique PCODEs: ['SS01' 'SS02' 'SS03' 'SS04' 'SS05' 'SS06' 'SS07' 'SS08' 'SS09' 'SS10']


In [7]:
# Search the ENTIRE South Sudan dataframe for "abyei" regardless of adm_level
abyei_rows = rainfall_ss[
    rainfall_ss.astype(str)
    .apply(lambda row: row.str.contains("abyei", case=False))
    .any(axis=1)
]

print("Found Abyei rows:", len(abyei_rows))
if len(abyei_rows) > 0:
    print(abyei_rows[["adm_level", "PCODE"]].drop_duplicates())

Found Abyei rows: 0


In [1]:
import processing.rainfall_processing as rain

df, rainfall_predictors = rain.get_clean_data(False, False)

INFO:hdx.api.configuration:No HDX base configuration parameter. Using default base configuration file: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml.
INFO:hdx.api.configuration:Loading HDX base configuration from: C:\Users\evely\PycharmProjects\msc-final-project\.venv\Lib\site-packages\hdx\api\hdx_base_configuration.yaml
INFO:hdx.api.configuration:No HDX configuration parameter and no configuration file at default path: C:\Users\evely\.hdx_configuration.yaml.
INFO:hdx.api.configuration:Read only access to HDX: True
INFO:Hdx Ingest:download=False: Reading local file ../data/hdx\sdn-rainfall-subnat-full.csv


##  Test GDF

In [4]:
data_name = "cod-ab-sdn"

In [8]:
hdx = HdxClient()
gdf = hdx.get_admin_boundaries(
    dataset_name=data_name, file_name="admin1", file_type="geojson", download=False
)

INFO:WFP Ingest:download=False: Reading local file ../data/hdx\sdn_admin1.geojson


In [ ]:
acled = pd.read_csv("../data/all_data.csv")

In [ ]:
country_name_mapping = {"Abyei PCA": "Abyei", "Aj Jazirah": "Al Jazirah"}

gdf["adm1_name"] = gdf["adm1_name"].replace(country_name_mapping)

In [ ]:
gdf.columns

In [ ]:
# hdx = HdxClient()
# df_sudan = hdx.get_data(dataset_name="wfp-food-prices-for-sudan", file_name="Sudan - Food Prices", file_type='csv')
# df_south_sudan = hdx.get_data(dataset_name="wfp-food-prices-for-south-sudan", file_name="South Sudan - Food Prices", file_type='csv')
#
# # https://www.britannica.com/place/Sudan/Agriculture-forestry-and-fishing
# primary_commodities = ["Sorghum", "Millet", "Wheat flour"]
#
# df_filtered = df[
#     (df["commodity"].isin(primary_commodities))
#     & (df["pricetype"] == "Retail")
#     & (df["priceflag"] == "actual")
# ].copy()
#
# model_df = df_filtered[
#     ["date", "admin1", "admin2", "commodity", "unit", "usdprice"]
# ]
#
# def clean_state_names(value):
#     key = str(value).strip()
#     return SUDAN_STATE_MAPPING.get(key, value)  # Returns original if not matched
#
#
# renamed_df = model_df.copy()
# renamed_df["admin1"] = model_df["admin1"].apply(clean_state_names)
#
# # Now both are DataFrames!
# find_unmatched_unique_values(acled, renamed_df, "admin1")

In [ ]:
# hdx = HdxClient()
# df_sudan = hdx.get_data(dataset_name="wfp-food-prices-for-sudan", file_name="Sudan - Food Prices", file_type='csv')
# df_south_sudan = hdx.get_data(dataset_name="wfp-food-prices-for-south-sudan", file_name="South Sudan - Food Prices",
#                               file_type='csv')

In [ ]:
# df_south_sudan_abyei = df_south_sudan[
#     df_south_sudan["admin1"] == "Abyei"]
#
# df_combined = pd.concat([df_sudan, df_south_sudan_abyei], ignore_index=True)
#
# # https://www.britannica.com/place/Sudan/Agriculture-forestry-and-fishing
# primary_commodities = ["Sorghum", "Millet", "Wheat flour"]
#
# df_filtered = df_combined[
#     (df_combined["commodity"].isin(primary_commodities))
#     & (df_combined["pricetype"] == "Retail")
#     & (df_combined["priceflag"] == "actual")
# ].copy()
#
# model_df = df_filtered[
#     ["date", "admin1", "admin2", "commodity", "unit", "usdprice"]
# ].copy()
#
# def clean_state_names(value):
#     if pd.isna(value):
#         return value
#     key = str(value).strip()
#     return SUDAN_STATE_MAPPING.get(key, value)
#
# renamed_df = model_df.copy()
# renamed_df["admin1"] = renamed_df["admin1"].apply(clean_state_names)

In [6]:
def find_unmatched_unique_values(df1, df2, column_name):
    # Ensure we are working with a 1D Series even if duplicate columns exist
    s1 = df1[column_name]
    if isinstance(s1, pd.DataFrame):
        s1 = s1.iloc[:, 0]

    s2 = df2[column_name]
    if isinstance(s2, pd.DataFrame):
        s2 = s2.iloc[:, 0]

    set1 = set(s1.dropna().unique())
    set2 = set(s2.dropna().unique())

    only_in_df1 = set1 - set2
    only_in_df2 = set2 - set1

    print(f"Only in first dataframe: {only_in_df1}")
    print(f"Only in second dataframe: {only_in_df2}")
    return only_in_df1, only_in_df2

In [9]:
df

,admin1,year_month,rainfall_3m_anomaly
0,Al Jazirah,2017-07,0.000000
1,Al Jazirah,2017-08,133.196290
2,Al Jazirah,2017-09,116.572365
3,Al Jazirah,2017-10,112.880300
4,Al Jazirah,2017-11,108.707060
...,...,...,...
1615,White Nile,2024-08,119.630200
1616,White Nile,2024-09,116.999080
1617,White Nile,2024-10,111.128426
1618,White Nile,2024-11,107.648660


In [8]:
acled = pd.read_csv("../data/all_data.csv")
df = df.rename(columns={"region": "admin1"})
find_unmatched_unique_values(acled, df, "admin1")

Only in first dataframe: {'Abyei'}
Only in second dataframe: set()


({'Abyei'}, set())

In [2]:
SUDAN_STATE_MAPPING = {
    "Al Gezira": "Al Jazirah",
    "Nile": "River Nile",
    "Eastern Darfur": "East Darfur",
    "Abyei PCA": "Abyei",
    "Aj Jazirah": "Al Jazirah",
}

In [13]:
def read_food_prices():
    hdx = HdxClient()
    df_sudan = hdx.get_data(
        dataset_name="wfp-food-prices-for-sudan",
        file_name="Sudan - Food Prices",
        file_type="csv",
    )
    df_south_sudan = hdx.get_data(
        dataset_name="wfp-food-prices-for-south-sudan",
        file_name="South Sudan - Food Prices",
        file_type="csv",
    )

    df_abyei = df_south_sudan[df_south_sudan["market"] == "Abyei"].copy()
    df_abyei["admin1"] = "Abyei"

    df_combined = pd.concat([df_sudan, df_abyei], ignore_index=True)

    primary_commodities = ["Sorghum", "Millet", "Wheat flour"]

    df_filtered = df_combined[
        (df_combined["commodity"].isin(primary_commodities))
        & (df_combined["pricetype"] == "Retail")
        & (df_combined["priceflag"].isin(["actual", "aggregate"]))
    ].copy()

    df_filtered_cols = df_filtered[
        ["date", "admin1", "admin2", "market", "commodity", "unit", "usdprice"]
    ].copy()

    def clean_state_names(value):
        return SUDAN_STATE_MAPPING.get(value, value)

    renamed_df = df_filtered_cols.copy()
    renamed_df["admin1"] = df_filtered_cols["admin1"].apply(clean_state_names)
    renamed_df["date"] = pd.to_datetime(renamed_df["date"])
    renamed_df["year_month"] = renamed_df["date"].dt.to_period("M")
    renamed_df["unit_weight_in_kg"] = (
        renamed_df["unit"]
        .str.extract(r"([\d.]+)", expand=False)
        .astype(float)
        .fillna(1.0)
    )
    renamed_df["usdprice"] = pd.to_numeric(renamed_df["usdprice"], errors="coerce")
    renamed_df["usdprice_per_kg"] = (
        renamed_df["usdprice"] / renamed_df["unit_weight_in_kg"]
    )
    return renamed_df

In [14]:
food_prices_df = read_food_prices()

Downloaded file to: ..\data\hdx\Sudan - Food Prices7.csv
Successfully loaded 'Sudan - Food Prices' into variable dataframe
Downloaded file to: ..\data\hdx\South Sudan - Food Prices2.csv
Successfully loaded 'South Sudan - Food Prices' into variable dataframe


In [15]:
food_prices_df

,date,admin1,admin2,market,commodity,unit,usdprice,year_month,unit_weight_in_kg,usdprice_per_kg
190,2003-01-15,East Darfur,Ed Daein,Eddein,Millet,3.5 KG,0.66,2003-01,3.5,0.188571
193,2003-01-15,Kassala,Kassala,Kassala,Millet,3.5 KG,0.40,2003-01,3.5,0.114286
194,2003-01-15,Kassala,Kassala,Kassala,Sorghum,3 KG,0.47,2003-01,3.0,0.156667
198,2003-01-15,North Darfur,El Fasher,Al Fashir,Millet,3.5 KG,0.77,2003-01,3.5,0.220000
199,2003-01-15,North Darfur,El Fasher,Al Fashir,Sorghum,3 KG,0.68,2003-01,3.0,0.226667
...,...,...,...,...,...,...,...,...,...,...
22130,2026-01-15,Abyei,Twic,Abyei,Wheat flour,KG,2.01,2026-01,1.0,2.010000
22147,2026-02-15,Abyei,Twic,Abyei,Wheat flour,KG,1.78,2026-02,1.0,1.780000
22166,2026-03-15,Abyei,Twic,Abyei,Wheat flour,KG,1.78,2026-03,1.0,1.780000
22186,2026-06-15,Abyei,Twic,Abyei,Wheat flour,KG,2.16,2026-06,1.0,2.160000


In [16]:
def process_and_pivot_food_prices(df_prices: pd.DataFrame) -> pd.DataFrame:
    df = df_prices.copy()

    df_grouped = (
        df.groupby(["admin1", "year_month", "commodity"])["usdprice_per_kg"]
        .median()
        .reset_index()
    )

    all_regions = df_grouped["admin1"].unique()
    all_months = pd.period_range(
        df_grouped["year_month"].min(), df_grouped["year_month"].max(), freq="M"
    )
    all_commodities = df_grouped["commodity"].unique()

    full_index = pd.MultiIndex.from_product(
        [all_regions, all_months, all_commodities],
        names=["admin1", "year_month", "commodity"],
    )

    df_expanded = (
        df_grouped.set_index(["admin1", "year_month", "commodity"])
        .reindex(full_index)
        .reset_index()
        .sort_values(["admin1", "commodity", "year_month"])
    )

    df_expanded["usdprice_per_kg"] = df_expanded.groupby(["admin1", "commodity"])[
        "usdprice_per_kg"
    ].transform(lambda x: x.ffill().bfill())

    df_pivoted = df_expanded.pivot(
        index=["admin1", "year_month"],
        columns="commodity",
        values="usdprice_per_kg",
    ).reset_index()

    df_pivoted.columns.name = None
    df_pivoted = df_pivoted.rename(
        columns={
            "Millet": "price_millet",
            "Sorghum": "price_sorghum",
            "Wheat flour": "price_wheat_flour",
        }
    )

    return df_pivoted

In [17]:
pivoted = process_and_pivot_food_prices(food_prices_df)

In [18]:
pivoted

,admin1,year_month,price_millet,price_sorghum,price_wheat_flour
0,Abyei,2003-01,NaN,NaN,1.55
1,Abyei,2003-02,NaN,NaN,1.55
2,Abyei,2003-03,NaN,NaN,1.55
3,Abyei,2003-04,NaN,NaN,1.55
4,Abyei,2003-05,NaN,NaN,1.55
...,...,...,...,...,...
5372,White Nile,2026-03,3.211429,2.066667,4.17
5373,White Nile,2026-04,3.331429,2.500000,5.71
5374,White Nile,2026-05,3.748571,2.620000,5.56
5375,White Nile,2026-06,3.882857,2.290000,5.55


In [ ]:
# df_abyei = df_south_sudan[df_south_sudan["market"] == "Abyei"].copy()
# df_abyei["admin1"] = "Abyei"
#
# df_combined = pd.concat([df_sudan, df_abyei], ignore_index=True)
#
# primary_commodities = ["Sorghum", "Millet", "Wheat flour"]
#
# df_filtered = df_combined[
#     (df_combined["commodity"].isin(primary_commodities))
#     & (df_combined["pricetype"] == "Retail")
#     & (df_combined["priceflag"].isin(["actual", "aggregate"]))
# ].copy()
#
# model_df = df_filtered[
#     ["date", "admin1", "admin2", "market", "commodity", "unit", "usdprice"]
# ].copy()
#
# SUDAN_STATE_MAPPING = {"Al Gezira": "Al Jazirah",
#  "Nile": "River Nile",
#  "Eastern Darfur": "East Darfur"}
#
#
# def clean_state_names(value):
#     return SUDAN_STATE_MAPPING.get(value, value)
#
#
# renamed_df = model_df.copy()
# renamed_df["admin1"] = renamed_df["admin1"].apply(clean_state_names)
#
# unmatched = find_unmatched_unique_values(acled, renamed_df, "admin1")
#
# print("States only in ACLED:", unmatched["only_in_df1"])
# print("States only in WFP Food Prices:", unmatched["only_in_df2"])

In [ ]:
renamed_df["date"] = pd.to_datetime(renamed_df["date"])
renamed_df["year_month"] = renamed_df["date"].dt.to_period("M")

In [ ]:
renamed_df